# Libraries

In [1]:
# Install required packages
!pip install segmentation-models-pytorch rasterio scikit-learn scipy -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 2.9 MB/s eta 0:00:0000:01


## Step 1 - Download Sen1Floods11 Imagery

In [1]:
import os, subprocess

def setup_sen1floods11_pipeline(base_dir="./data/sen1floods11"):
    os.makedirs(base_dir, exist_ok=True)
    gs_root = "gs://sen1floods11/v1.1/data/flood_events/HandLabeled/"
    layers  = ["S1Hand", "S2Hand", "LabelHand"]
    print(f"--- Downloading Sen1Floods11 HandLabeled to {base_dir} ---")
    for layer in layers:
        os.makedirs(os.path.join(base_dir, layer), exist_ok=True)
        cmd = f"gsutil -m cp -r -n {gs_root}{layer} {base_dir}"
        try:
            subprocess.run(cmd, shell=True, check=True)
            print(f"  OK  {layer}")
        except subprocess.CalledProcessError:
            print(f"  FAIL {layer}")
    for layer in layers:
        path  = os.path.join(base_dir, layer)
        count = len(os.listdir(path)) if os.path.exists(path) else 0
        print(f"  {layer}: {count} chips")

setup_sen1floods11_pipeline()


--- Downloading Sen1Floods11 HandLabeled to ./data/sen1floods11 ---


Skipping existing item: file://./data/sen1floods11/S1Hand/Bolivia_103757_S1Hand.tif
Skipping existing item: file://./data/sen1floods11/S1Hand/Bolivia_23014_S1Hand.tif
Skipping existing item: file://./data/sen1floods11/S1Hand/Bolivia_129334_S1Hand.tif
Skipping existing item: file://./data/sen1floods11/S1Hand/Bolivia_195474_S1Hand.tif
Skipping existing item: file://./data/sen1floods11/S1Hand/Bolivia_233925_S1Hand.tif
Skipping existing item: file://./data/sen1floods11/S1Hand/Bolivia_242570_S1Hand.tif
Skipping existing item: file://./data/sen1floods11/S1Hand/Bolivia_360519_S1Hand.tif
Skipping existing item: file://./data/sen1floods11/S1Hand/Ghana_103272_S1Hand.tif
Skipping existing item: file://./data/sen1floods11/S1Hand/Bolivia_290290_S1Hand.tif
Skipping existing item: file://./data/sen1floods11/S1Hand/Bolivia_294583_S1Hand.tif
Skipping existing item: file://./data/sen1floods11/S1Hand/Ghana_134751_S1Hand.tif
Skipping existing item: file://./data/sen1floods11/S1Hand/Bolivia_379434_S1Hand.t

  OK  S1Hand


Skipping existing item: file://./data/sen1floods11/S2Hand/Bolivia_23014_S2Hand.tif
Skipping existing item: file://./data/sen1floods11/S2Hand/Bolivia_129334_S2Hand.tif
Skipping existing item: file://./data/sen1floods11/S2Hand/Bolivia_103757_S2Hand.tif
Skipping existing item: file://./data/sen1floods11/S2Hand/Bolivia_195474_S2Hand.tif
Skipping existing item: file://./data/sen1floods11/S2Hand/Bolivia_233925_S2Hand.tif
Skipping existing item: file://./data/sen1floods11/S2Hand/Ghana_124834_S2Hand.tif
Skipping existing item: file://./data/sen1floods11/S2Hand/Ghana_103272_S2Hand.tif
Skipping existing item: file://./data/sen1floods11/S2Hand/Bolivia_242570_S2Hand.tif
Skipping existing item: file://./data/sen1floods11/S2Hand/Bolivia_290290_S2Hand.tif
Skipping existing item: file://./data/sen1floods11/S2Hand/Bolivia_294583_S2Hand.tif
Skipping existing item: file://./data/sen1floods11/S2Hand/Bolivia_312675_S2Hand.tif
Skipping existing item: file://./data/sen1floods11/S2Hand/Bolivia_379434_S2Hand.t

  OK  S2Hand


Skipping existing item: file://./data/sen1floods11/LabelHand/Bolivia_103757_LabelHand.tif
Skipping existing item: file://./data/sen1floods11/LabelHand/Bolivia_129334_LabelHand.tif
Skipping existing item: file://./data/sen1floods11/LabelHand/Bolivia_195474_LabelHand.tif
Skipping existing item: file://./data/sen1floods11/LabelHand/Bolivia_23014_LabelHand.tif
Skipping existing item: file://./data/sen1floods11/LabelHand/Bolivia_233925_LabelHand.tif
Skipping existing item: file://./data/sen1floods11/LabelHand/Bolivia_432776_LabelHand.tif
Skipping existing item: file://./data/sen1floods11/LabelHand/Bolivia_379434_LabelHand.tif
Skipping existing item: file://./data/sen1floods11/LabelHand/Bolivia_242570_LabelHand.tif
Skipping existing item: file://./data/sen1floods11/LabelHand/Bolivia_290290_LabelHand.tif
Skipping existing item: file://./data/sen1floods11/LabelHand/Bolivia_294583_LabelHand.tif
Skipping existing item: file://./data/sen1floods11/LabelHand/Bolivia_360519_LabelHand.tif
Skipping ex

  OK  LabelHand
  S1Hand: 446 chips
  S2Hand: 446 chips
  LabelHand: 446 chips


## Step 2 - Preprocessing: Build 10-Channel Feature Tensors
**New in v7:** Adds SWIR1 (B11) and MNDWI channels.
MNDWI outperforms NDWI for flood/built-up discrimination (Li et al. 2025).

In [4]:
import os, glob
import numpy as np
import rasterio
import pandas as pd
from tqdm import tqdm

def build_sen1floods_tensors(base_dir="./data/sen1floods11",
                              output_dir="./data/ml_dataset"):
    """
    10-CHANNEL FEATURE TENSOR (MANZAR v7)
    ─────────────────────────────────────
    Ch  0  VV copol         SAR dB → [0,1]
    Ch  1  VH crosspol      SAR dB → [0,1]
    Ch  2  Blue   B02       /10000
    Ch  3  Green  B03       /10000
    Ch  4  Red    B04       /10000
    Ch  5  NIR    B08       /10000
    Ch  6  NDWI             (Green-NIR)/(Green+NIR)
    Ch  7  SAR ratio        VV/VH
    Ch  8  SWIR1  B11       /10000   < NEW: water content signal
    Ch  9  MNDWI            (Green-SWIR1)/(Green+SWIR1) < NEW: better water index

    Why SWIR? Li et al. (2025) show +3-5% IoU gain on Sen1Floods11 by
    adding SWIR bands. MNDWI outperforms NDWI for distinguishing flood
    water from built-up areas and wet soil.
    """
    s1_dir  = os.path.join(base_dir, 'S1Hand')
    s2_dir  = os.path.join(base_dir, 'S2Hand')
    lbl_dir = os.path.join(base_dir, 'LabelHand')

    feat_dir = os.path.join(output_dir, 'features')
    mask_dir = os.path.join(output_dir, 'masks')
    os.makedirs(feat_dir, exist_ok=True)
    os.makedirs(mask_dir, exist_ok=True)

    s1_files = glob.glob(os.path.join(s1_dir, '*.tif'))
    if not s1_files:
        raise FileNotFoundError(f"No S1 files in {s1_dir}. Run pipeline cell first.")

    manifest = []
    print(f"Processing {len(s1_files)} triplets → 10-channel tensors...")

    for s1_path in tqdm(s1_files, desc="Preprocessing"):
        filename  = os.path.basename(s1_path)
        base_name = filename.replace('_S1Hand.tif', '')
        s2_path   = os.path.join(s2_dir,  f"{base_name}_S2Hand.tif")
        lbl_path  = os.path.join(lbl_dir, f"{base_name}_LabelHand.tif")

        if not (os.path.exists(s2_path) and os.path.exists(lbl_path)):
            continue

        with rasterio.open(s1_path) as src:  s1 = src.read()   # [2, 512, 512]
        with rasterio.open(s2_path) as src:  s2 = src.read()   # [13,512, 512]
        with rasterio.open(lbl_path) as src: lbl= src.read(1)  # [512, 512]

        # ── SAR calibration ──────────────────────────────────────────
        s1_db   = 10 * np.log10(np.where(s1 > 0, s1, 1e-7))
        s1_norm = np.clip((s1_db + 30) / 30.0, 0, 1)
        copol, crosspol = s1_norm[0], s1_norm[1]

        # ── Optical normalisation ─────────────────────────────────────
        # Sentinel-2 S2Hand band order: B01,B02,B03,B04,B05,B06,B07,B08,B8A,B09,B10,B11,B12
        blue  = np.clip(s2[1]  / 10000.0, 0, 1)   # B02
        green = np.clip(s2[2]  / 10000.0, 0, 1)   # B03
        red   = np.clip(s2[3]  / 10000.0, 0, 1)   # B04
        nir   = np.clip(s2[7]  / 10000.0, 0, 1)   # B08
        swir1 = np.clip(s2[11] / 10000.0, 0, 1)   # B11 < NEW

        # ── Derived indices ───────────────────────────────────────────
        with np.errstate(divide='ignore', invalid='ignore'):
            ndwi  = np.where((green+nir)   > 0, (green-nir)  /(green+nir),   0.0)
            mndwi = np.where((green+swir1) > 0, (green-swir1)/(green+swir1), 0.0)
            sar_r = np.where(crosspol > 0,       copol/crosspol,             0.0)

        ndwi  = np.nan_to_num(ndwi,  nan=0., posinf= 1., neginf=-1.)
        mndwi = np.nan_to_num(mndwi, nan=0., posinf= 1., neginf=-1.)
        sar_r = np.nan_to_num(sar_r, nan=0., posinf= 0., neginf= 0.)

        # ── 10-channel tensor ─────────────────────────────────────────
        features = np.stack(
            [copol, crosspol, blue, green, red, nir, ndwi, sar_r, swir1, mndwi],
            axis=0
        ).astype(np.float32)   # [10, 512, 512]
        mask     = lbl.astype(np.int8)

        patch_id = f"SEN1_{base_name}"
        np.save(os.path.join(feat_dir, f"{patch_id}.npy"), features)
        np.save(os.path.join(mask_dir, f"{patch_id}.npy"), mask)

        valid_px = np.sum(mask != -1)
        water_px = np.sum(mask == 1)
        manifest.append({
            "patch_id"          : patch_id,
            "biome"             : base_name.split('_')[0],
            "water_ratio"       : round(water_px / valid_px, 4) if valid_px > 0 else 0.0,
            "valid_pixel_ratio" : round(valid_px / (512*512),  4),
        })

    df = pd.DataFrame(manifest)
    df.to_csv(os.path.join(output_dir, 'patch_manifest.csv'), index=False)
    print(f"\n=== Preprocessing complete ===")
    print(f"  Tensors   : {len(df)}  (10-channel, 512x512)")
    print(f"  Water+    : {(df['water_ratio']>0.01).sum()}")
    print(f"  Channels  : VV VH Blue Green Red NIR NDWI SARratio SWIR1 MNDWI")
    print(f"  Manifest  : {output_dir}/patch_manifest.csv")

build_sen1floods_tensors()


Processing 446 triplets → 10-channel tensors...


Preprocessing: 100%|██████████| 446/446 [00:21<00:00, 20.94it/s]


=== Preprocessing complete ===
  Tensors   : 446  (10-channel, 512x512)
  Water+    : 293
  Channels  : VV VH Blue Green Red NIR NDWI SARratio SWIR1 MNDWI
  Manifest  : ./data/ml_dataset/patch_manifest.csv


## Step 3 - Download Official Sen1Floods11 Split CSVs
**REQUIRED.** Raises `FileNotFoundError` if unavailable.
These define the exact patches used in Bonafilia 2020 and Yadav 2022.

In [10]:
import os
import subprocess
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
GCS_BASE = "gs://sen1floods11/v1.1/splits/flood_handlabeled"

PATHS = {
    "train": "./flood_train_data.csv",
    "test":  "./flood_test_data.csv",
    "val":   "./flood_val_data.csv"
}

RANDOM_SEED = 42
VAL_RATIO = 0.1

# ─────────────────────────────────────────────────────────────
# STEP 1: DOWNLOAD
# ─────────────────────────────────────────────────────────────
def download(remote, local):
    if os.path.exists(local):
        print(f"[OK] {local}")
        return

    print(f"[DOWNLOADING] {remote}")
    r = subprocess.run(["gsutil", "cp", remote, local],
                       capture_output=True, text=True)

    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError("Download failed")


print("\n=== STEP 1: DOWNLOAD ===")
for split in ["train", "test"]:
    download(f"{GCS_BASE}/flood_{split}_data.csv", PATHS[split])

# ─────────────────────────────────────────────────────────────
# STEP 2: EVENT-BASED SPLIT (CORRECT)
# ─────────────────────────────────────────────────────────────
print("\n=== STEP 2: EVENT-BASED SPLIT ===")

if not os.path.exists(PATHS["val"]):

    df = pd.read_csv(PATHS["train"])

    # detect path column automatically
    path_col = df.columns[0]

    # ✅ robust event extraction
    def extract_event(path):
        filename = os.path.basename(path)   # e.g. Bolivia_103757_S1.tif
        event = filename.split("_")[0]      # → Bolivia
        return event

    df["event"] = df[path_col].apply(extract_event)

    # sanity check
    print("\n[DEBUG] Unique events:", df["event"].unique())
    print("[DEBUG] Event counts:\n", df["event"].value_counts())

    # group split (NO leakage)
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=VAL_RATIO,
        random_state=RANDOM_SEED
    )

    train_idx, val_idx = next(splitter.split(df, groups=df["event"]))

    train_df = df.iloc[train_idx].drop(columns=["event"])
    val_df   = df.iloc[val_idx].drop(columns=["event"])

    # overwrite train (important for consistency)
    train_df.to_csv(PATHS["train"], index=False)
    val_df.to_csv(PATHS["val"], index=False)

    print(f"\n[DONE] Train: {len(train_df)}")
    print(f"[DONE] Val:   {len(val_df)}")

else:
    print("[SKIP] Val already exists")

# ─────────────────────────────────────────────────────────────
# FINAL CHECK
# ─────────────────────────────────────────────────────────────
print("\n=== FINAL CHECK ===")
for k, v in PATHS.items():
    print(f"{k}: {os.path.exists(v)}")

print("\n✔ Dataset ready (TRUE paper-aligned)")


=== STEP 1: DOWNLOAD ===
[OK] ./flood_train_data.csv
[OK] ./flood_test_data.csv

=== STEP 2: EVENT-BASED SPLIT ===

[DEBUG] Unique events: ['Ghana' 'India' 'Mekong' 'Nigeria' 'Pakistan' 'Paraguay' 'Somalia'
 'Spain' 'Sri-Lanka' 'USA']
[DEBUG] Event counts:
 event
USA          41
India        40
Paraguay     39
Ghana        30
Sri-Lanka    24
Mekong       18
Spain        18
Pakistan     16
Somalia      15
Nigeria      10
Name: count, dtype: int64

[DONE] Train: 227
[DONE] Val:   24

=== FINAL CHECK ===
train: True
test: True
val: True

✔ Dataset ready (TRUE paper-aligned)


## Step 4 - MANZAR v7 Training
Set `TRAINING_MODE` then run:
- `"standard"` - single run, full model (start here)
- `"ablation"` - all 7 variants systematically
- `"crossval"` - 5-fold CV on official train patches

In [3]:

import os, gc, csv, random, shutil, logging, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms.functional as TF
import segmentation_models_pytorch as smp
from sklearn.model_selection import KFold
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm import tqdm

# ─────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────
MANIFEST_PATH      = "./data/ml_dataset/patch_manifest.csv"
FEATURES_DIR       = "./data/ml_dataset/features"
MASKS_DIR          = "./data/ml_dataset/masks"
CHECKPOINT_DIR     = "./checkpoints"
BEST_WEIGHTS_ROOT  = "./best_flood_model_{variant}.pth"

# Official split CSVs - REQUIRED. Raises error if missing.
OFFICIAL_TRAIN_CSV = "./flood_train_data.csv"
OFFICIAL_VAL_CSV   = "./flood_val_data.csv"

# ── Training mode ──────────────────────────────────────────────────
# "standard" - single run, full model, official split
# "ablation" - trains all 7 variants systematically
# "crossval" - 5-fold CV on official training patches
TRAINING_MODE    = "standard"
ABLATION_VARIANT = "full"   # used only when TRAINING_MODE="standard"

CV_FOLDS = 5
CV_SEED  = 42

# ── Core hyperparameters ───────────────────────────────────────────
SEED             = 42
EPOCHS           = 250
PHYSICAL_BATCH   = 8
ACCUM_STEPS      = 4        # effective batch = 32
NUM_WORKERS      = 4
LR               = 2e-4
WEIGHT_DECAY     = 1e-3
ETA_MIN          = 1e-6
PATIENCE         = 25
VAL_SMOOTHING    = 0.3

ALPHA_BCE        = 0.3
FT_ALPHA         = 0.6      # FP penalty > 0.5 → penalise false alarms more
FT_BETA          = 0.4
FT_GAMMA         = 4/3

OVERSAMPLE_THRESH = 0.05
OVERSAMPLE_WEIGHT = 2.0

OPT_DROP_START   = 0.25
OPT_DROP_END     = 0.05
OPT_DROP_RAMP    = 60

THRESH_SWEEP     = [0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70]
IGNORE_INDEX     = -1

# ─────────────────────────────────────────────────────────────────────
# LOGGING
# ─────────────────────────────────────────────────────────────────────
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(os.path.join(CHECKPOINT_DIR,"train.log")),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger("manzar_v7")


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def save_hyperparams(variant="full"):
    # Saves full hyperparameter table to JSON for reproducibility
    params = {
        "version"         : "MANZAR v7",
        "TRAINING_MODE"   : TRAINING_MODE,
        "ABLATION_VARIANT": variant,
        "INPUT_CHANNELS"  : 10,
        "CHANNEL_LAYOUT"  : "VV VH Blue Green Red NIR NDWI SARratio SWIR1 MNDWI",
        "EPOCHS"          : EPOCHS,
        "PHYSICAL_BATCH"  : PHYSICAL_BATCH,
        "ACCUM_STEPS"     : ACCUM_STEPS,
        "EFFECTIVE_BATCH" : PHYSICAL_BATCH * ACCUM_STEPS,
        "LR"              : LR,
        "WEIGHT_DECAY"    : WEIGHT_DECAY,
        "ETA_MIN"         : ETA_MIN,
        "PATIENCE"        : PATIENCE,
        "VAL_SMOOTHING_EMA": VAL_SMOOTHING,
        "LOSS"            : "HybridLoss: 0.7*FocalTversky + 0.3*BCE",
        "FT_ALPHA"        : FT_ALPHA,
        "FT_BETA"         : FT_BETA,
        "FT_GAMMA"        : round(FT_GAMMA,4),
        "OVERSAMPLE_WEIGHT": OVERSAMPLE_WEIGHT,
        "OPT_DROP_START"  : OPT_DROP_START,
        "OPT_DROP_END"    : OPT_DROP_END,
        "OPT_DROP_RAMP"   : OPT_DROP_RAMP,
        "SEED"            : SEED,
        "SPLIT"           : "OFFICIAL Sen1Floods11",
        "TRAIN_CSV"       : OFFICIAL_TRAIN_CSV,
        "VAL_CSV"         : OFFICIAL_VAL_CSV,
    }
    path = os.path.join(CHECKPOINT_DIR, f"hyperparams_{variant}.json")
    with open(path,"w") as f: json.dump(params, f, indent=2)
    log.info("="*62)
    log.info("  MANZAR v7 - HYPERPARAMETER TABLE")
    log.info("="*62)
    for k,v in params.items(): log.info(f"  {k:<22} : {v}")
    log.info(f"  Saved: {path}")
    log.info("="*62)


# ─────────────────────────────────────────────────────────────────────
# 1. DATA SPLIT  - OFFICIAL ONLY (no fallback)
# ─────────────────────────────────────────────────────────────────────
def _check_official_csvs():
    missing = []
    for p in [OFFICIAL_TRAIN_CSV, OFFICIAL_VAL_CSV]:
        if not os.path.exists(p): missing.append(p)
    if missing:
        raise FileNotFoundError(
            f"Official split CSVs not found: {missing}\n"
            "These are REQUIRED for benchmark-comparable results.\n"
            "Run the 'Download Official Splits' cell above first."
        )


def _parse_official_ids(csv_path):
    df = pd.read_csv(csv_path, header=None)
    ids = set()
    for val in df.iloc[:,0]:
        base = str(val).replace("S1Hand/","").replace("_S1Hand.tif","")
        ids.add(f"SEN1_{base}")
    return ids


def make_official_splits(manifest_path):
    _check_official_csvs()
    df = pd.read_csv(manifest_path)
    if "biome" not in df.columns:
        df["biome"] = df["patch_id"].apply(
            lambda x: x.split("_")[1] if "_" in x else "unknown")

    train_ids = _parse_official_ids(OFFICIAL_TRAIN_CSV)
    val_ids   = _parse_official_ids(OFFICIAL_VAL_CSV)
    train_df  = df[df["patch_id"].isin(train_ids)].reset_index(drop=True)
    val_df    = df[df["patch_id"].isin(val_ids)].reset_index(drop=True)
    unmatched = len(df) - len(train_df) - len(val_df)

    log.info("="*62)
    log.info("  SPLIT: OFFICIAL Sen1Floods11")
    log.info(f"  Train     : {len(train_df)} patches")
    log.info(f"  Val       : {len(val_df)} patches")
    log.info(f"  Test (held): {unmatched} patches (Bolivia event - never touched)")
    log.info(f"  Val biomes: {sorted(val_df['biome'].unique().tolist())}")
    log.info(f"  Val water+: {(val_df['water_ratio']>0.01).sum()}/{len(val_df)}")
    log.info("="*62)

    if len(train_df) == 0:
        raise RuntimeError(
            "Official split matched 0 training patches. "
            f"Check patch_id format.\n"
            f"  Manifest: {df['patch_id'].tolist()[:3]}\n"
            f"  CSV     : {list(train_ids)[:3]}"
        )
    return train_df, val_df


def make_kfold_splits(manifest_path, n_folds=5, seed=42):
    # 5-fold CV on official TRAIN patches only
    _check_official_csvs()
    df = pd.read_csv(manifest_path)
    if "biome" not in df.columns:
        df["biome"] = df["patch_id"].apply(
            lambda x: x.split("_")[1] if "_" in x else "unknown")
    train_ids = _parse_official_ids(OFFICIAL_TRAIN_CSV)
    cv_df = df[df["patch_id"].isin(train_ids)].reset_index(drop=True)
    log.info(f"CV pool: {len(cv_df)} official train patches, {n_folds}-fold")
    kf    = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = []
    for fold_i,(tr,vl) in enumerate(kf.split(cv_df)):
        folds.append((cv_df.iloc[tr].reset_index(drop=True),
                      cv_df.iloc[vl].reset_index(drop=True), fold_i+1))
    return folds


# ─────────────────────────────────────────────────────────────────────
# 2. DATASET
# ─────────────────────────────────────────────────────────────────────
class FloodDataset(Dataset):
    def __init__(self, df, features_dir, masks_dir, is_train=True, opt_drop_prob=0.0):
        self.df=df; self.fd=features_dir; self.md=masks_dir
        self.is_train=is_train; self.odp=opt_drop_prob

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        pid = self.df.iloc[idx]["patch_id"]
        x   = torch.from_numpy(np.load(os.path.join(self.fd,f"{pid}.npy"))).float()
        y   = torch.from_numpy(np.load(os.path.join(self.md,f"{pid}.npy"))).long()

        if self.is_train:
            if random.random()>0.5:
                x=TF.hflip(x); y=TF.hflip(y.unsqueeze(0)).squeeze(0)
            if random.random()>0.5:
                x=TF.vflip(x); y=TF.vflip(y.unsqueeze(0)).squeeze(0)
            k=random.choice([0,1,2,3])
            if k>0: x=torch.rot90(x,k,[1,2]); y=torch.rot90(y,k,[0,1])
            # Optical dropout: zero channels 2-9 (all optical+derived)
            if random.random()<self.odp: x[2:,:,:]=0.0
            # SAR speckle simulation
            if random.random()<0.30:
                x[:2]=(x[:2]+torch.randn_like(x[:2])*0.02).clamp(0,1)
        return x, y


def build_train_loader(df, batch_size, num_workers, opt_drop_prob):
    w = np.where(df["water_ratio"]>OVERSAMPLE_THRESH, OVERSAMPLE_WEIGHT, 1.0)
    s = WeightedRandomSampler(w, num_samples=len(w), replacement=True)
    d = FloodDataset(df, FEATURES_DIR, MASKS_DIR, is_train=True, opt_drop_prob=opt_drop_prob)
    return DataLoader(d, batch_size=batch_size, sampler=s,
                      num_workers=num_workers, pin_memory=True, drop_last=True)

def build_val_loader(df, num_workers):
    d = FloodDataset(df, FEATURES_DIR, MASKS_DIR, is_train=False)
    return DataLoader(d, batch_size=16, shuffle=False,
                      num_workers=num_workers, pin_memory=True)


# ─────────────────────────────────────────────────────────────────────
# 3. ARCHITECTURE  (10-channel input)
# ─────────────────────────────────────────────────────────────────────
class CBAM(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.avg=nn.AdaptiveAvgPool2d(1); self.max=nn.AdaptiveMaxPool2d(1)
        self.fc =nn.Sequential(
            nn.Linear(channels,channels//reduction,bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels//reduction,channels,bias=False))
        self.sc =nn.Conv2d(2,1,kernel_size=7,padding=3,bias=False)

    def forward(self, x):
        b,c,_,_=x.shape
        a=self.fc(self.avg(x).view(b,c)); m=self.fc(self.max(x).view(b,c))
        x=x*torch.sigmoid(a+m).view(b,c,1,1)
        s=torch.cat([x.mean(1,keepdim=True),x.max(1,keepdim=True)[0]],dim=1)
        return x*torch.sigmoid(self.sc(s))


class DualStreamUNetPP(nn.Module):
    # MANZAR v7: 10-ch dual-stream. SAR=[0,1,7] 3-ch. Opt=[2,3,4,5,6,8,9] 7-ch. Fusion->CBAM->head.
    def __init__(self, use_cbam=True):
        super().__init__()
        self.use_cbam    = use_cbam
        self.sar_encoder = smp.UnetPlusPlus(
            encoder_name="resnet50", encoder_weights="imagenet",
            in_channels=3, classes=32)
        self.opt_encoder = smp.UnetPlusPlus(
            encoder_name="resnet50", encoder_weights="imagenet",
            in_channels=7, classes=32)   # < 7 instead of 5 (v6)
        if use_cbam: self.cbam = CBAM(64, reduction=8)
        self.head = nn.Sequential(
            nn.Conv2d(64,32,3,padding=1,bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Dropout2d(p=0.15), nn.Conv2d(32,1,1))

    def forward(self, x):
        sar = torch.cat([x[:,0:2,:,:], x[:,7:8,:,:]], dim=1)      # ch 0,1,7
        opt = torch.cat([x[:,2:7,:,:], x[:,8:10,:,:]], dim=1)     # ch 2-6, 8-9
        f   = torch.cat([self.sar_encoder(sar), self.opt_encoder(opt)], dim=1)
        if self.use_cbam: f = self.cbam(f)
        return self.head(f)


class SingleStreamSAR(nn.Module):
    # Ablation baseline: SAR only, 3-channel
    def __init__(self):
        super().__init__()
        self.enc = smp.UnetPlusPlus(
            encoder_name="resnet50", encoder_weights="imagenet",
            in_channels=3, classes=1)

    def forward(self, x):
        return self.enc(torch.cat([x[:,0:2,:,:],x[:,7:8,:,:]],dim=1))


class SingleStreamSAR2ch(nn.Module):
    # Ablation: SAR VV+VH only, 2-channel
    def __init__(self):
        super().__init__()
        self.enc = smp.UnetPlusPlus(
            encoder_name="resnet50", encoder_weights="imagenet",
            in_channels=2, classes=1)

    def forward(self, x): return self.enc(x[:,0:2,:,:])


def build_model(variant="full"):
    if variant=="full":              return DualStreamUNetPP(use_cbam=True)
    elif variant=="no_cbam":         return DualStreamUNetPP(use_cbam=False)
    elif variant in ("no_optical","single_stream_sar"): return SingleStreamSAR()
    elif variant=="no_sar_ratio":    return SingleStreamSAR2ch()
    # These variants use full architecture; their change is in loss/dropout
    elif variant in ("no_opt_dropout","symmetric_loss","no_swir"):
        return DualStreamUNetPP(use_cbam=True)
    else: raise ValueError(f"Unknown variant: {variant!r}")


# ─────────────────────────────────────────────────────────────────────
# 4. LOSS
# ─────────────────────────────────────────────────────────────────────
class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=FT_ALPHA, beta=FT_BETA, gamma=FT_GAMMA, smooth=1e-6):
        super().__init__()
        self.a=alpha; self.b=beta; self.g=gamma; self.s=smooth

    def forward(self, logits, targets):
        p=torch.sigmoid(logits).view(-1); t=targets.view(-1)
        v=t!=IGNORE_INDEX; p=p[v]; t=t[v].float()
        TP=(p*t).sum(); FP=((1-t)*p).sum(); FN=(t*(1-p)).sum()
        T=(TP+self.s)/(TP+self.a*FP+self.b*FN+self.s)
        return (1-T)**self.g


class HybridLoss(nn.Module):
    def __init__(self, alpha_bce=ALPHA_BCE, ft_alpha=FT_ALPHA, ft_beta=FT_BETA):
        super().__init__()
        self.ft=FocalTverskyLoss(alpha=ft_alpha,beta=ft_beta)
        self.bce=nn.BCEWithLogitsLoss(reduction="none"); self.a=alpha_bce

    def forward(self, logits, targets):
        fl=self.ft(logits,targets)
        vm=(targets!=IGNORE_INDEX).float()
        bm=self.bce(logits.squeeze(1), targets.float().clamp(0,1))
        bl=(bm*vm).sum()/(vm.sum()+1e-8)
        return (1-self.a)*fl + self.a*bl


def get_criterion(variant):
    if variant=="symmetric_loss":
        return HybridLoss(alpha_bce=ALPHA_BCE, ft_alpha=0.5, ft_beta=0.5)
    return HybridLoss()


# ─────────────────────────────────────────────────────────────────────
# 5. METRICS
# ─────────────────────────────────────────────────────────────────────
class RunningMetrics:
    def __init__(self): self.reset()
    def reset(self): self.TP=self.TN=self.FP=self.FN=0

    def update(self, probs, targets, threshold=0.5):
        p=probs.view(-1); t=targets.view(-1); v=t!=IGNORE_INDEX
        p=(p[v]>threshold).long(); t=t[v].long()
        self.TP+=((p==1)&(t==1)).sum().item()
        self.TN+=((p==0)&(t==0)).sum().item()
        self.FP+=((p==1)&(t==0)).sum().item()
        self.FN+=((p==0)&(t==1)).sum().item()

    def compute(self):
        e=1e-8; tp,tn,fp,fn=self.TP,self.TN,self.FP,self.FN
        pr=tp/(tp+fp+e); re=tp/(tp+fn+e)
        return {"accuracy":(tp+tn)/(tp+tn+fp+fn+e),"precision":pr,"recall":re,
                "f1":2*pr*re/(pr+re+e),"iou":tp/(tp+fp+fn+e)}


# ─────────────────────────────────────────────────────────────────────
# 6. CHECKPOINTS
# ─────────────────────────────────────────────────────────────────────
def save_ckpt(path, epoch, model, opt, sched, best, smooth, patience):
    torch.save({"epoch":epoch,"model_state":model.state_dict(),
                "optimizer_state":opt.state_dict(),"scheduler_state":sched.state_dict(),
                "best_val_iou":best,"smoothed_val_iou":smooth,
                "epochs_no_improve":patience}, path)

def load_ckpt(path, model, opt, sched, device):
    c=torch.load(path,map_location=device)
    s={k.replace("module.",""):v for k,v in c["model_state"].items()}
    model.load_state_dict(s); opt.load_state_dict(c["optimizer_state"])
    sched.load_state_dict(c["scheduler_state"])
    log.info(f"Resumed epoch {c['epoch']} | best={c['best_val_iou']:.4f}")
    return c["epoch"],c["best_val_iou"],c.get("smoothed_val_iou",c["best_val_iou"]),c["epochs_no_improve"]


# ─────────────────────────────────────────────────────────────────────
# 7. TRAIN / VAL LOOPS
# ─────────────────────────────────────────────────────────────────────
def opt_drop(epoch, variant):
    if variant=="no_opt_dropout": return 0.0
    if epoch>=OPT_DROP_RAMP: return OPT_DROP_END
    return OPT_DROP_START+(epoch/OPT_DROP_RAMP)*(OPT_DROP_END-OPT_DROP_START)


def train_epoch(model, loader, crit, opt, scaler, device, ep):
    model.train(); rl=0.; m=RunningMetrics(); opt.zero_grad()
    bar=tqdm(loader,desc=f"Train {ep}",leave=False,dynamic_ncols=True)
    for step,(x,y) in enumerate(bar):
        x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
        with torch.amp.autocast("cuda"):
            logits=model(x); loss=crit(logits,y)/ACCUM_STEPS
        scaler.scale(loss).backward()
        if (step+1)%ACCUM_STEPS==0 or (step+1)==len(loader):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad()
        rl+=loss.item()*ACCUM_STEPS
        with torch.no_grad():
            m.update(torch.sigmoid(logits.detach()).squeeze(1).cpu(),y.cpu())
        bar.set_postfix(loss=f"{loss.item()*ACCUM_STEPS:.4f}")
    r=m.compute(); r["loss"]=rl/len(loader); return r


@torch.no_grad()
def val_epoch(model, loader, crit, device):
    model.eval(); rl=0.; ap,at=[],[]
    for x,y in tqdm(loader,desc="Val",leave=False,dynamic_ncols=True):
        x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
        with torch.amp.autocast("cuda"):
            logits=model(x); rl+=crit(logits,y).item()
        ap.append(torch.sigmoid(logits).squeeze(1).cpu()); at.append(y.cpu())
    ap=torch.cat(ap); at=torch.cat(at)
    bf,bt,bm=0.0,0.5,None
    for t in THRESH_SWEEP:
        m=RunningMetrics(); m.update(ap,at,t); cm=m.compute()
        if cm["f1"]>bf: bf,bt,bm=cm["f1"],t,cm
    if bm is None:
        m=RunningMetrics(); m.update(ap,at,0.5); bm,bt=m.compute(),0.5
    bm["loss"]=rl/len(loader); bm["thresh"]=bt; return bm


# ─────────────────────────────────────────────────────────────────────
# 8. SINGLE TRAINING RUN
# ─────────────────────────────────────────────────────────────────────
def train_single(train_df, val_df, variant="full", suffix="", resume=None):
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log.info(f"\n{'='*62}\n  RUN  variant={variant}  suffix={suffix or 'none'}")
    log.info(f"  Train={len(train_df)}  Val={len(val_df)}\n{'='*62}")

    best_path = os.path.join(CHECKPOINT_DIR, f"best_{variant}{suffix}.pth")
    model=build_model(variant)
    if torch.cuda.device_count()>1: model=nn.DataParallel(model)
    model=model.to(device)
    log.info(f"  Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    opt   =optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    sched =optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS,eta_min=ETA_MIN)
    scaler=torch.amp.GradScaler("cuda")
    crit  =get_criterion(variant)

    se=0; bv=0.0; sv=0.0; en=0
    if resume and os.path.exists(resume):
        se,bv,sv,en=load_ckpt(resume,model,opt,sched,device)

    val_loader=build_val_loader(val_df,NUM_WORKERS)
    logf=open(os.path.join(CHECKPOINT_DIR,f"log_{variant}{suffix}.csv"),"w",newline="")
    w=csv.writer(logf)
    w.writerow(["epoch","lr","opt_drop","train_loss","train_f1","train_iou",
                "val_loss","val_prec","val_rec","val_f1","val_iou","val_thresh","smooth_iou"])

    pd_prev=None; tl=None
    for epoch in range(se,EPOCHS):
        lr  =opt.param_groups[0]["lr"]
        drop=opt_drop(epoch,variant)
        if tl is None or abs(drop-(pd_prev or 999))>0.01:
            tl=build_train_loader(train_df,PHYSICAL_BATCH,NUM_WORKERS,drop)
            pd_prev=drop

        tm=train_epoch(model,tl,crit,opt,scaler,device,epoch+1)
        vm=val_epoch(model,val_loader,crit,device)
        sched.step()

        vi=vm["iou"]
        sv=vi if sv==0. else (1-VAL_SMOOTHING)*sv+VAL_SMOOTHING*vi
        log.info(f"  Ep{epoch+1:>3} lr={lr:.1e} TL={tm['loss']:.4f} TF1={tm['f1']:.4f} "
                 f"VL={vm['loss']:.4f} VF1={vm['f1']:.4f} VIoU={vi:.4f} "
                 f"Sm={sv:.4f} Thr={vm['thresh']:.2f} Drop={drop:.2f}")
        w.writerow([epoch+1,lr,drop,tm["loss"],tm["f1"],tm["iou"],
                    vm["loss"],vm["precision"],vm["recall"],vm["f1"],vi,vm["thresh"],sv])
        logf.flush()

        if sv>bv:
            bv=sv; en=0
            torch.save(model.state_dict(),best_path)
            shutil.copy2(best_path, f"./best_flood_model_{variant}{suffix}.pth")
            log.info(f"  >> Best Sm={bv:.4f} (raw={vi:.4f}) saved")
        else: en+=1

        if (epoch+1)%10==0:
            save_ckpt(os.path.join(CHECKPOINT_DIR,f"ckpt_{epoch+1:04d}_{variant}{suffix}.pth"),
                      epoch+1,model,opt,sched,bv,sv,en)
        if en>=PATIENCE:
            log.info(f"  Early stop epoch {epoch+1}"); break

    logf.close()
    log.info(f"  DONE  variant={variant}  best_sm_iou={bv:.4f}")
    return bv


# ─────────────────────────────────────────────────────────────────────
# 9. ABLATION VARIANTS
# ─────────────────────────────────────────────────────────────────────
ABLATION_VARIANTS = [
    ("single_stream_sar",  "Baseline: SAR-only single-stream UNet++ (no optical, no CBAM)"),
    ("no_sar_ratio",       "SAR VV+VH only - no VV/VH ratio channel"),
    ("no_optical",         "SAR stream only with ratio - tests optical contribution"),
    ("symmetric_loss",     "Full dual-stream, symmetric loss α=β=0.5 - tests FP penalty"),
    ("no_cbam",            "Dual-stream, no CBAM - tests attention contribution"),
    ("no_opt_dropout",     "Full dual-stream, no optical dropout curriculum"),
    ("full",               "Full MANZAR v7 - all components enabled"),
]


def run_ablation(train_df, val_df):
    log.info("\n"+"="*62+"\n  ABLATION STUDY - 7 variants on official split\n"+"="*62)
    results={}
    for variant,desc in ABLATION_VARIANTS:
        log.info(f"\n  [{variant}] {desc}")
        iou=train_single(train_df,val_df,variant=variant)
        results[variant]={"description":desc,"val_iou":round(iou,4)}
        log.info(f"  RESULT [{variant}]: {iou:.4f}")
    log.info("\n"+"="*62+"\n  ABLATION SUMMARY (ascending val IoU)")
    for v,r in sorted(results.items(),key=lambda x: x[1]["val_iou"]):
        log.info(f"  {v:<22} : {r['val_iou']:.4f} - {r['description'][:45]}")
    log.info("="*62)
    with open(os.path.join(CHECKPOINT_DIR,"ablation_val_results.json"),"w") as f:
        json.dump(results,f,indent=2)
    log.info(f"  Saved: {CHECKPOINT_DIR}/ablation_val_results.json")
    return results


def run_crossval(manifest_path, variant="full"):
    log.info(f"\n{'='*62}\n  {CV_FOLDS}-FOLD CV  variant={variant}\n{'='*62}")
    folds=make_kfold_splits(manifest_path,CV_FOLDS,CV_SEED)
    ious=[]
    for tr,vl,fi in folds:
        log.info(f"\n  FOLD {fi}/{CV_FOLDS}: train={len(tr)} val={len(vl)}")
        iou=train_single(tr,vl,variant=variant,suffix=f"_fold{fi}")
        ious.append(iou)
        log.info(f"  FOLD {fi}: {iou:.4f}")
    mean,std=np.mean(ious),np.std(ious)
    log.info(f"\n  CV RESULT: {mean:.4f} ± {std:.4f}")
    log.info(f"  Per-fold : {[round(x,4) for x in ious]}")
    with open(os.path.join(CHECKPOINT_DIR,f"cv_{variant}.json"),"w") as f:
        json.dump({"folds":ious,"mean":round(mean,4),"std":round(std,4)},f,indent=2)
    return mean,std


# ─────────────────────────────────────────────────────────────────────
# 10. MAIN
# ─────────────────────────────────────────────────────────────────────
def main():
    set_seed(SEED)
    os.makedirs(CHECKPOINT_DIR,exist_ok=True)
    torch.cuda.empty_cache(); gc.collect()
    save_hyperparams(variant=ABLATION_VARIANT)

    if TRAINING_MODE=="standard":
        train_df,val_df=make_official_splits(MANIFEST_PATH)
        train_single(train_df,val_df,variant=ABLATION_VARIANT)

    elif TRAINING_MODE=="ablation":
        train_df,val_df=make_official_splits(MANIFEST_PATH)
        run_ablation(train_df,val_df)

    elif TRAINING_MODE=="crossval":
        run_crossval(MANIFEST_PATH,variant=ABLATION_VARIANT)

    else:
        raise ValueError(f"Unknown TRAINING_MODE: {TRAINING_MODE!r}")

main()


2026-03-20 01:38:21,121 | INFO | ==============================================================
2026-03-20 01:38:21,122 | INFO |   MANZAR v7 - HYPERPARAMETER TABLE
2026-03-20 01:38:21,123 | INFO | ==============================================================
2026-03-20 01:38:21,123 | INFO |   version                : MANZAR v7
2026-03-20 01:38:21,124 | INFO |   TRAINING_MODE          : standard
2026-03-20 01:38:21,125 | INFO |   ABLATION_VARIANT       : full
2026-03-20 01:38:21,126 | INFO |   INPUT_CHANNELS         : 10
2026-03-20 01:38:21,126 | INFO |   CHANNEL_LAYOUT         : VV VH Blue Green Red NIR NDWI SARratio SWIR1 MNDWI
2026-03-20 01:38:21,127 | INFO |   EPOCHS                 : 250
2026-03-20 01:38:21,128 | INFO |   PHYSICAL_BATCH         : 8
2026-03-20 01:38:21,129 | INFO |   ACCUM_STEPS            : 4
2026-03-20 01:38:21,130 | INFO |   EFFECTIVE_BATCH        : 32
2026-03-20 01:38:21,130 | INFO |   LR                     : 0.0002
2026-03-20 01:38:21,131 | INFO |   WEIGHT_DE

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

2026-03-20 01:38:22,056 | INFO | HTTP Request: HEAD https://huggingface.co/smp-hub/resnet50.imagenet/resolve/00cb74e366966d59cd9a35af57e618af9f88efe9/model.safetensors "HTTP/1.1 302 Found"
2026-03-20 01:38:22,158 | INFO | HTTP Request: GET https://huggingface.co/api/models/smp-hub/resnet50.imagenet/xet-read-token/00cb74e366966d59cd9a35af57e618af9f88efe9 "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

2026-03-20 01:38:24,820 | INFO |   Params: 98,012,675
2026-03-20 01:39:11,507 | INFO |   Ep  1 lr=2.0e-04 TL=0.7276 TF1=0.2815 VL=0.7514 VF1=0.8354 VIoU=0.7174 Sm=0.7174 Thr=0.55 Drop=0.25
2026-03-20 01:39:12,210 | INFO |   >> Best Sm=0.7174 (raw=0.7174) saved
2026-03-20 01:39:56,560 | INFO |   Ep  2 lr=2.0e-04 TL=0.6460 TF1=0.5525 VL=0.6477 VF1=0.8596 VIoU=0.7537 Sm=0.7283 Thr=0.55 Drop=0.25
2026-03-20 01:39:57,841 | INFO |   >> Best Sm=0.7283 (raw=0.7537) saved
2026-03-20 01:40:44,937 | INFO |   Ep  3 lr=2.0e-04 TL=0.6179 TF1=0.5774 VL=0.5944 VF1=0.8023 VIoU=0.6698 Sm=0.7108 Thr=0.70 Drop=0.24
2026-03-20 01:41:34,238 | INFO |   Ep  4 lr=2.0e-04 TL=0.6155 TF1=0.5617 VL=0.5599 VF1=0.8860 VIoU=0.7954 Sm=0.7361 Thr=0.60 Drop=0.24
2026-03-20 01:41:35,552 | INFO |   >> Best Sm=0.7361 (raw=0.7954) saved
2026-03-20 01:42:23,665 | INFO |   Ep  5 lr=2.0e-04 TL=0.5904 TF1=0.6657 VL=0.5526 VF1=0.8769 VIoU=0.7808 Sm=0.7496 Thr=0.70 Drop=0.24
2026-03-20 01:42:24,953 | INFO |   >> Best Sm=0.7496 (r

RuntimeError: [enforce fail at inline_container.cc:664] . unexpected pos 648331264 vs 648331152

In [ ]:
!rm -rf /kaggle/working/checkpoints
print("Done")